In [1]:
#목적 , 목표 구현하고자하는 서비스 | 부석하고자 하는 주제 
#수단, 도구 Desk, Field, Crawling 
#요즘 사람들은 어떤 책을 좋아할까? 정보 수집 데이터 분석 또 다른 서비스 런칭 
#고객들에게 제공해야하는 기초 정보 
# 책이름, 저자, 출판사, 가격, 할인, 재고, 목차, 줄거리, 카테고리 

#단계를 분할해서 크롤링 > 디버깅 
# 주요카테고리 3개 정도만 정보 수집 
# 주요 카테고리 3개에 따른 책이름, 저자, 출판사, 가격, 할인, 재고, 목차, 줄거리, 카테고리

In [ ]:
import requests 
from bs4 import BeautifulSoup
import time

headers = {
    "User-Agent" : 
'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Mobile Safari/537.36'
    
}
category_urls = {
    "웹툰" : "https://www.yes24.com/product/search?domain=BOOK&query=%25EC%259B%25B9%25ED%2588%25B0&dispNo2=001001008&page=1&size=24&dispNo3=001001008020",
    "로맨스":"https://www.yes24.com/product/search?domain=BOOK&query=%25EC%259B%25B9%25ED%2588%25B0&dispNo2=001001008&page=1&size=24&dispNo3=001001008016",
    "국내도서_자기계발": "https://www.yes24.com/product/search?domain=BOOK&query=%25EC%259B%25B9%25ED%2588%25B0&dispNo2=001001008&page=1&size=24&dispNo3=001001008012"
}

books = []

def crawl_category(category_name, category_url, max_count = 2) :
    count = 0;

    while count < max_count :
        resp = requests.get(category_url, headers = headers)
        soup = BeautifulSoup(resp.text, "html.parser")

        book_links = soup.select("#yesSchList > li:nth-child(1) > div > div.item_info > div.info_row.info_name > a.gd_name")
        

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import json

headers = {
    "User-Agent": 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Mobile Safari/537.36'

# 모바일 검색 URL
category_urls = {
    "웹툰": "https://m.yes24.com/Search?domain=BOOK&query=웹툰&page=",
    "로맨스": "https://m.yes24.com/Search?domain=BOOK&query=로맨스&page=",
    "자기계발": "https://m.yes24.com/Search?domain=BOOK&query=자기계발&page="
}

books = []


# =============================
# 1) 리뷰 크롤링 (모바일 JSON API)
# =============================
def crawl_reviews(goods_no, max_pages=10):
    reviews = []

    for page in range(1, max_pages + 1):
        url = f"https://m.yes24.com/Product/communityModules/GoodsReviewList/{goods_no}?PageNumber={page}"
        resp = requests.get(url, headers=headers)

        try:
            data = resp.json()
        except:
            break

        review_html = data.get("Html", None)
        if not review_html:
            break

        soup = BeautifulSoup(review_html, "html.parser")
        review_items = soup.select(".review_cont")

        if not review_items:
            break

        for r in review_items:
            review_text = r.get_text(strip=True)
            reviews.append(review_text)

        time.sleep(0.5)

    return reviews


# =============================
# 2) 상세페이지 크롤링
# =============================
def crawl_book_detail(book_url):
    resp = requests.get(book_url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    title = soup.select_one("h1.gd_name")
    title = title.get_text(strip=True) if title else None

    author = soup.select_one(".gd_auth a")
    author = author.get_text(strip=True) if author else None

    publisher = soup.select_one(".gd_pub")
    publisher = publisher.get_text(strip=True) if publisher else None

    price = soup.select_one(".yes_mypage_price em")
    price = price.get_text(strip=True) if price else None

    desc_tag = soup.select_one("#infoset_contents")
    description = desc_tag.get_text(strip=True) if desc_tag else ""

    goods_no = book_url.split("/")[-1]

    reviews = crawl_reviews(goods_no)

    return {
        "title": title,
        "author": author,
        "publisher": publisher,
        "price": price,
        "description": description,
        "review_count": len(reviews),
        "reviews": reviews,
        "url": book_url
    }


# =============================
# 3) 검색 결과 페이지 크롤링
# =============================
def crawl_category(category_name, base_url):
    print(f"\n📌 카테고리 시작: {category_name}")

    for page in range(1, 11):
        url = f"{base_url}{page}"
        print(f"▶ 검색 페이지: {url}")

        resp = requests.get(url, headers=headers)
        soup = BeautifulSoup(resp.text, "html.parser")

        book_links = soup.select("ul#schList li a.gd_name")

        if not book_links:
            print(" ❌ 더 이상 책 없음 → 종료")
            break

        for a in book_links:
            book_url = "https://m.yes24.com" + a["href"]
            print(f"   → 상세페이지: {book_url}")

            try:
                detail = crawl_book_detail(book_url)
                detail["category"] = category_name
                books.append(detail)
            except Exception as e:
                print("⚠ 오류 발생:", e)

            time.sleep(1)


# =============================
# 4) 실행
# =============================
for name, url in category_urls.items():
    crawl_category(name, url)


# =============================
# 5) 엑셀 저장
# =============================
rows = []
for b in books:
    if b["reviews"]:
        for rev in b["reviews"]:
            rows.append({
                "category": b["category"],
                "title": b["title"],
                "author": b["author"],
                "publisher": b["publisher"],
                "price": b["price"],
                "description": b["description"],
                "review": rev,
                "url": b["url"]
            })
    else:
        rows.append({
            "category": b["category"],
            "title": b["title"],
            "author": b["author"],
            "publisher": b["publisher"],
            "price": b["price"],
            "description": b["description"],
            "review": "",
            "url": b["url"]
        })

df = pd.DataFrame(rows)
df.to_excel("yes24_mobile_books.xlsx", index=False)

print("\n🎉 저장 완료: yes24_mobile_books.xlsx")



📌 카테고리 시작: 웹툰
▶ 검색 페이지: https://m.yes24.com/Search?domain=BOOK&query=웹툰&page=1
 ❌ 더 이상 책 없음 → 종료

📌 카테고리 시작: 로맨스
▶ 검색 페이지: https://m.yes24.com/Search?domain=BOOK&query=로맨스&page=1
 ❌ 더 이상 책 없음 → 종료

📌 카테고리 시작: 자기계발
▶ 검색 페이지: https://m.yes24.com/Search?domain=BOOK&query=자기계발&page=1
 ❌ 더 이상 책 없음 → 종료

🎉 저장 완료: yes24_mobile_books.xlsx


In [34]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE = "https://www.yes24.com"

headers = {
    "User-Agent": "Mozilla/5.0"
}

def get_books(page):
    url = f"{BASE}/Product/Search?query=웹툰&domain=BOOK&page={page}"
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    items = soup.select("ul#yesSchList li")

    books = []

    for li in items:
        a = li.select_one("a.gd_name")
        if not a:
            continue

        href = a["href"]
        title = a.get_text(strip=True)

        # /Product/Goods/147807114
        goods_no = href.split("/")[-1]

        books.append({
            "title": title,
            "goods_no": goods_no,
            "url": BASE + href
        })

    return books


def get_reviews(goods_no):
    reviews = []
    for page in range(1, 20):  # 최대 20페이지
        api = f"https://www.yes24.com/Product/Goods/Review/{goods_no}?PageNumber={page}"
        res = requests.get(api, headers=headers)

        try:
            data = res.json()
        except:
            break

        arr = data.get("ReviewList", [])
        if not arr:
            break

        for r in arr:
            reviews.append(r.get("ReviewContent", "").strip())

        time.sleep(0.7)

    return reviews


all_books = []
for p in range(1, 4):
    print("📄 페이지:", p)
    books = get_books(p)
    print("  → 수집된 상품:", len(books))

    for b in books:
        print("    리뷰:", b["title"])
        b["reviews"] = get_reviews(b["goods_no"])
        all_books.append(b)

df = pd.DataFrame(all_books)
df.to_excel("yes24_webtoon_pc.xlsx", index=False)

print("🎉 저장 완료!")


📄 페이지: 1
  → 수집된 상품: 24
    리뷰: 만화 나 혼자만 레벨업 15
    리뷰: 44교시 생존수업 1~2 세트
    리뷰: 산타 스카우트 : 크리스마스 대작전
    리뷰: 오래 보고 싶었다
    리뷰: 킬러 배드로 5~6 세트
    리뷰: 만화웹툰 장르 대백과
    리뷰: 나는 웹툰강사다
    리뷰: 엄마를 만나러 가는 길 1
    리뷰: 만화 인소의 법칙 8 한정판
    리뷰: 만화 나 혼자만 레벨업 14
    리뷰: 엄마를 만나러 가는 길 2
    리뷰: 엄마를 만나러 가는 길 1~2권 세트
    리뷰: 킬러 배드로 6
    리뷰: 킬러 배드로 5
    리뷰: 행복아, 어서 와
    리뷰: 시든 꽃에 눈물을 1
    리뷰: 소꿉친구 컴플렉스 세트
    리뷰: 시든 꽃에 눈물을 2
    리뷰: 시든 꽃에 눈물을 1~2권 세트
    리뷰: 웹툰 만화시집 3권 세트
    리뷰: AI 메이커 교사가 만든 AI 아트디렉터를 위한 찐 실전 챗GPT 생성형 AI 창의 융합 교육 - AI 웹툰·동화책 만들기/AI 작곡하기
    리뷰: 시든 꽃에 눈물을 3
    리뷰: 만화 나 혼자만 레벨업 1~5 박스세트
    리뷰: 시든 꽃에 눈물을 4
📄 페이지: 2
  → 수집된 상품: 24
    리뷰: 만화 나 혼자만 레벨업 11~15 박스세트
    리뷰: 만화 인소의 법칙 8
    리뷰: 별을 사랑하여
    리뷰: 만화 나 혼자만 레벨업 1~15 박스 세트
    리뷰: 만화 나 혼자만 레벨업 13
    리뷰: 만화 나 혼자만 레벨업 10
    리뷰: 만화 나 혼자만 레벨업 6~10 박스세트
    리뷰: 만화 나 혼자만 레벨업 15 한정판
    리뷰: 만화 나 혼자만 레벨업 12
    리뷰: 만화 나 혼자만 레벨업 11
    리뷰: 숲속의 담 1 : 자라지 않는 소년
    리뷰: 스토리텔링 우동이즘의 잘 팔리는 웹툰, 웹소설 이야기 만들기
    리뷰: 웹툰 캐릭터 그리기 대작전
    리뷰: 생성형

In [35]:
import requests
from bs4 import BeautifulSoup
import time
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
}
CATEGORY_URLS = {
    "국내도서_경제경영": "https://www.yes24.com/Product/Category/Display/001001025",
    "국내도서_IT": "https://www.yes24.com/product/category/display/001001003",
    "국내도서_자기계발": "https://www.yes24.com/Product/Category/Display/001001026"
}
books = []
def parse_book_detail(detail_url, category_name) :
    resp = requests.get(detail_url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")
    title = soup.select_one("h2.gd_name").get_text(strip=True) if soup.select_one("h2.gd_name") else None
    author = soup.select_one("span.gd_pubArea span.gd_auth").get_text(strip=True) if soup.select_one("span.gd_pubArea span.gd_auth") else None
    publisher = soup.select_one("span.gd_pub a").get_text(strip=True) if soup.select_one("span.gd_pub a") else None
    list_price = soup.select_one("span.nor_price em.yes_m").get_text(strip=True) if soup.select_one("span.nor_price em.yes_m") else None
    sale_price = soup.select_one("span.crema_price em.yes_m").get_text(strip=True) if soup.select_one("span.crema_price em.yes_m") else None
    thumb = soup.select_one("img.gImg")["src"] if soup.select_one("img.gImg") else None
    summary = soup.select_one("div.infoWrap_txtInner > textarea.txtContentText").get_text(strip=True) if soup.select_one("div.infoWrap_txtInner > textarea.txtContentText") else None
    toc = soup.select_one("div.infoWrap_txt > textarea.txtContentText").get_text(strip=True) if soup.select_one("div.infoWrap_txt > textarea.txtContentText") else None
    sale_state_elem = soup.select_one("p.gd_saleState em")
    stock = "정보없음"
    if sale_state_elem :
        state_text = sale_state_elem.get_text(strip=True)
        if "판매중" in state_text :
            stock = "재고있음"
        elif "일시품절" in state_text or "품절" in state_text :
            stock = "품절"
        else :
            stock = state_text
    books.append({
        "category": category_name,
        "title": title,
        "author": author,
        "publisher": publisher,
        "list_price": list_price,
        "sale_price": sale_price,
        "thumbnail": thumb,
        "summary": summary,
        "toc": toc,
        "detail_url": detail_url,
        "stock": stock
    })
def crawl_category(category_name, category_url, max_count = 2) :
    count = 0;
    while count < max_count :
        resp = requests.get(category_url, headers=headers)
        soup = BeautifulSoup(resp.text, "html.parser")
        book_links = soup.select("li.item div.item_info > div.info_row.info_name > a")
        if not book_links :
            break
        for a in book_links :
            detail_url = "https://www.yes24.com" + a["href"]
            parse_book_detail(detail_url, category_name)
            count += 1
            if count >= max_count :
                break
            time.sleep(0.5)
for cat_name, cat_url in CATEGORY_URLS.items() :
    crawl_category(cat_name, cat_url, max_count = 2)
print(books)

[{'category': '국내도서_경제경영', 'title': '트렌드 코리아 2026', 'author': '김난도,전미영,최지혜,권정윤,한다혜저 외 7명정보 더 보기/감추기김난도전미영최지혜권정윤한다혜이혜원이수진서유현전다현이준영이향은김나은', 'publisher': '미래의창', 'list_price': '18,000', 'sale_price': '16,500', 'thumbnail': 'https://image.yes24.com/goods/153064968/XL', 'summary': 'HORSE POWERAI 대전환의 시대, 무엇을 준비해야 하는가?세상은 작용과 반작용, 치열한 정반합(正反合)의 소용돌이가 거세게 휘몰아치고 있다. 방향을 잡기 어려울 정도로 속도가 빠르고 정신이 없다. 그렇다면 우리의 방향타는 어디에 있는가? 거센 풍랑과 어디서 불어올지 모르는 태풍을 피하기 위해 우리는 어디에 닻을 내리고 있어야 하는가?관세전쟁과 특이점을 향하는 AI의 위협, 끝이 보이지 않는 전쟁의 소용돌이 속에서도 한국 경제는 자동차, 조선, 반도체에 이어 글로벌 시장에서 1위의 위업을 달성한 K뷰티, 더욱 한국적이 되어가는 K콘텐츠 열풍에 힘입어 아직은 순항을 이어가고 있다. 수많은 개인들이 보이는 역동적이고 창의적인 라이프스타일과 소비 행태들 역시 전에 없이 새롭고 흥미로운 트렌드를 만들어내고 있다. 배는 항구에 정박해 있을 때가 가장 안전하지만, 그것이 배의 본질은 아니다. 『트렌드 코리아』와 함께 2026년의 바다로 항해를 이어가자.', 'toc': '서문2026년 10대 소비트렌드 키워드1 · 2025 대한민국무경계 소비자얼어붙은 시장에 지펴진 새로운 불씨일상에 의미 더하기번아웃 시대 극복하기폭염이 만든 생존 경제, 기후가 시장을 삼키다〈트렌드 코리아〉 선정 2025년 대한민국 10대 트렌드 상품2 · 2026 트렌드휴먼인더루프 Human-in-the-loop필코노미 Oh, my feelings! The Feelconomy제로클릭 Results on Dem

In [37]:
import json

with open("books_yes24.json", "w", encoding="utf-8") as f :
    json.dump(books, f, ensure_ascii=False, )

🚀 YES24 만화 카테고리 전체 + 리뷰 수집 시작!
📄 상품 페이지 1 요청 중...


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [60]:
!pip install playwright playwright-stealth
!playwright install




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 MB 19.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [playwright-stealth]laywright-stealth]
136.3 MiB [                    ] 0% 0.0s136.3 MiB [                    ] 0% 81.0s136.3 MiB [                    ] 0% 163.6s136.3 MiB [                    ] 0% 94.9s136.3 MiB [                    ] 0% 96.2s136.3 MiB [                    ] 0% 69.5s136.3 MiB [                    ] 0% 65.3s136.3 MiB [                    ] 0% 47.9s136.3 MiB [                    ] 0% 40.5s136.3 MiB [                    ] 0% 36.3s136.3 MiB [                    ] 0% 24.6s136.3 MiB [                    ] 1% 21.1s136.3 MiB [                    ] 1% 21.2s136.3 MiB [                    ] 1% 23.0s136.3 MiB [                    ] 1% 20.2s136.3 MiB [                    ] 2% 15.9s136.3 MiB [=                   ] 2% 15.9s136.3 MiB [=                   ] 2% 17.0s136.3 MiB [=                   ] 2% 18.3s136.3 MiB [=                   ] 3% 15.9s136.

In [71]:
import time
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


BASE_URL = "https://www.yes24.com/Product/Category/Display/001001008020"
HEADERS = {"User-Agent": "Mozilla/5.0"}


# ---------------------------------------------------------
# 리스트 페이지 (requests)
# ---------------------------------------------------------
def get_list_page(page):
    url = BASE_URL + f"?page={page}"
    resp = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(resp.text, "html.parser")
    return soup.select("a.gd_name"), soup


def get_total_pages(soup):
    pages = []
    for a in soup.select("div.num_pg a"):
        txt = a.get_text(strip=True)
        if txt.isdigit():
            pages.append(int(txt))
    return max(pages) if pages else 1


# ---------------------------------------------------------
# Selenium Driver
# ---------------------------------------------------------
def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )
    return driver


# ---------------------------------------------------------
# 리뷰 수집 (페이지네이션 + 더보기 둘 다 완전지원)
# ---------------------------------------------------------
def fetch_reviews(driver):
    reviews = []

    time.sleep(1)

    # 1) 리뷰탭 클릭
    try:
        tab = driver.find_element(By.CSS_SELECTOR, "a#yReviewInfo")
        driver.execute_script("arguments[0].click();", tab)
        time.sleep(1.5)
    except:
        print("리뷰 탭 없음")
        return reviews

    # ------------------------
    # 내부 함수: 현재 페이지 리뷰 파싱
    # ------------------------
    def parse_current_reviews():
        soup = BeautifulSoup(driver.page_source, "html.parser")
        items = soup.select("div.review_cont")
        result = []

        for r in items:
            text = r.select_one("p.review_txt")
            text = text.get_text(strip=True) if text else ""

            user = r.select_one("span.txt_userid")
            user = user.get_text(strip=True) if user else ""

            date = r.select_one("span.txt_date")
            date = date.get_text(strip=True) if date else ""

            rating = None
            rt = r.select_one("span.rating")
            if rt:
                for cls in rt["class"]:
                    if "grade_" in cls:
                        rating = cls.replace("grade_", "")
                        break

            result.append({
                "user": user,
                "date": date,
                "rating": rating,
                "content": text
            })

        return result

    # ------------------------
    # 2) "더보기" 버튼 처리
    # ------------------------
    while True:
        reviews.extend(parse_current_reviews())

        try:
            more = driver.find_element(By.CSS_SELECTOR, "div.review_list_more a")
            driver.execute_script("arguments[0].click();", more)
            time.sleep(1.2)
        except:
            break

    # ------------------------
    # 3) 페이지네이션 처리
    # ------------------------
    while True:
        soup = BeautifulSoup(driver.page_source, "html.parser")

        current_pg = soup.select_one("div.review_paging strong")
        current_pg = int(current_pg.get_text(strip=True)) if current_pg else 1

        # 다음 페이지 버튼 존재 확인
        next_btn = None
        for btn in soup.select("div.review_paging a"):
            txt = btn.get_text(strip=True)
            if txt.isdigit() and int(txt) == current_pg + 1:
                next_btn = txt
                break

        if not next_btn:
            break

        try:
            btn_ele = driver.find_element(By.LINK_TEXT, str(current_pg + 1))
            driver.execute_script("arguments[0].click();", btn_ele)
            time.sleep(1.5)
        except:
            break

        reviews.extend(parse_current_reviews())

    return reviews


# ---------------------------------------------------------
# 상세 페이지 수집
# ---------------------------------------------------------
def parse_detail(driver, url):
    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    def sel(css):
        tag = soup.select_one(css)
        return tag.get_text(strip=True) if tag else ""

    title = sel("h2.gd_name")
    author = sel("span.gd_auth")
    publisher = sel("span.gd_pub a")
    list_price = sel("span.nor_price em.yes_m")
    sale_price = sel("span.salec_price em.yes_m")

    img = soup.select_one("img.gImg")
    img = img["src"] if img else ""

    summary = sel("textarea#infoset_introduce")
    toc = sel("textarea#infoset_toc")

    # 리뷰 수집
    reviews = fetch_reviews(driver)

    return {
        "title": title,
        "author": author,
        "publisher": publisher,
        "list_price": list_price,
        "sale_price": sale_price,
        "image": img,
        "summary": summary,
        "toc": toc,
        "url": url,
        "reviews": reviews,
    }


# ---------------------------------------------------------
# 전체 라노벨 크롤링
# ---------------------------------------------------------
def crawl_all():
    print("📌 1페이지 분석 중…")
    items, soup = get_list_page(1)
    total_pages = get_total_pages(soup)
    print(f"총 페이지: {total_pages}")

    driver = create_driver()
    results = []

    for page in range(1, total_pages + 1):
        print(f"\n===== {page}/{total_pages} 페이지 =====")
        items, _ = get_list_page(page)

        for a in items:
            detail_url = "https://www.yes24.com" + a["href"]
            print("상품:", detail_url)
            data = parse_detail(driver, detail_url)
            results.append(data)

    driver.quit()
    return results


# ---------------------------------------------------------
# CSV + JSON 저장
# ---------------------------------------------------------
def save_data(data):
    print("📁 저장 중…")

    # Books 데이터프레임
    books = []
    reviews = []

    for i, b in enumerate(data, start=1):
        book_id = i
        books.append({
            "book_id": book_id,
            "title": b["title"],
            "author": b["author"],
            "publisher": b["publisher"],
            "list_price": b["list_price"],
            "sale_price": b["sale_price"],
            "image": b["image"],
            "summary": b["summary"],
            "toc": b["toc"],
            "url": b["url"],
        })

        for r in b["reviews"]:
            reviews.append({
                "book_id": book_id,
                "review_user": r["user"],
                "review_date": r["date"],
                "review_rating": r["rating"],
                "review_content": r["content"],
            })

    # CSV 저장
    pd.DataFrame(books).to_csv("yes24_books.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(reviews).to_csv("yes24_reviews.csv", index=False, encoding="utf-8-sig")

    # JSON 저장
    with open("yes24_books.json", "w", encoding="utf-8") as f:
        json.dump(books, f, ensure_ascii=False, indent=2)

    with open("yes24_reviews.json", "w", encoding="utf-8") as f:
        json.dump(reviews, f, ensure_ascii=False, indent=2)

    print("🎉 CSV + JSON 저장 완료!")
    print(" - yes24_books.csv / yes24_reviews.csv")
    print(" - yes24_books.json / yes24_reviews.json")


# ---------------------------------------------------------
# 실행
# ---------------------------------------------------------
data = crawl_all()
save_data(data)


📌 1페이지 분석 중…
총 페이지: 1

===== 1/1 페이지 =====
상품: https://www.yes24.com/product/goods/153496863
리뷰 탭 없음
상품: https://www.yes24.com/product/goods/154297237
리뷰 탭 없음
상품: https://www.yes24.com/product/goods/161344815
리뷰 탭 없음
상품: https://www.yes24.com/product/goods/155111582
리뷰 탭 없음
상품: https://www.yes24.com/product/goods/159264348
리뷰 탭 없음
상품: https://www.yes24.com/product/goods/122676904
리뷰 탭 없음
상품: https://www.yes24.com/product/goods/164507418
리뷰 탭 없음
상품: https://www.yes24.com/product/goods/160112095


KeyboardInterrupt: 

In [75]:
import time
import json
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


# ==========================================================
# 1. Selenium Driver 설정
# ==========================================================
def get_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--window-size=1280,2000")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--ignore-certificate-errors")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver


# ==========================================================
# 2. 리뷰탭 열기
# ==========================================================
def open_review_tab(driver):
    # 리뷰 탭이 보이도록 충분히 스크롤
    for pos in [300, 700, 1100]:
        driver.execute_script(f"window.scrollTo(0, {pos});")
        time.sleep(0.5)

    # 리뷰/한줄평 탭 클릭 트리거(goGD_bot)
    try:
        driver.execute_script("goGD_bot(1);")
        time.sleep(1.2)
    except:
        pass


# ==========================================================
# 3. 전체 리뷰 탭 클릭
# ==========================================================
def click_total_review_tab(driver):
    try:
        elem = driver.find_element(By.CSS_SELECTOR, "#total > a > span")
        driver.execute_script("arguments[0].scrollIntoView(true);", elem)
        time.sleep(0.3)
        driver.execute_script("arguments[0].click();", elem)
        time.sleep(1.2)
        return True
    except Exception as e:
        print("🚨 전체 리뷰 탭 클릭 실패:", e)
        return False


# ==========================================================
# 4. 리뷰 리스트 파싱
# ==========================================================
def parse_reviews_from_dom(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.select("#infoset_reviewContentList li.reviewItem")

    result = []
    for li in items:
        content = li.select_one("p.review_txt")
        content = content.get_text(strip=True) if content else ""

        user = li.select_one(".txt_userid")
        user = user.get_text(strip=True) if user else ""

        date = li.select_one(".txt_date")
        date = date.get_text(strip=True) if date else ""

        rating = None
        rt = li.select_one("span.rating")
        if rt:
            for cls in rt.get("class", []):
                if "grade_" in cls:
                    rating = cls.replace("grade_", "")
                    break

        result.append({
            "user": user,
            "date": date,
            "rating": rating,
            "content": content,
        })

    return result


# ==========================================================
# 5. 리뷰 전체 페이지 크롤링
# ==========================================================
def fetch_all_reviews(driver):
    open_review_tab(driver)
    click_total_review_tab(driver)

    # 스크롤 내려서 리뷰 DOM 로딩
    for pos in [900, 1500, 2000]:
        driver.execute_script(f"window.scrollTo(0, {pos});")
        time.sleep(0.5)

    reviews = []
    reviews.extend(parse_reviews_from_dom(driver))

    while True:
        time.sleep(0.8)

        # 페이지네이션 존재 여부 확인
        try:
            paginator = driver.find_element(By.CSS_SELECTOR, ".yesUI_pagenS")
        except:
            break  # 페이지네이션 없음 → 리뷰 종료

        soup = BeautifulSoup(driver.page_source, "html.parser")

        try:
            current_page = int(soup.select_one(".yesUI_pagenS strong.num").text.strip())
        except:
            break

        next_btn = None
        for a in driver.find_elements(By.CSS_SELECTOR, ".yesUI_pagenS a.num"):
            if a.text.isdigit() and int(a.text) == current_page + 1:
                next_btn = a
                break

        if not next_btn:
            break

        # 다음 페이지 클릭 (JS 이벤트 포함)
        driver.execute_script("arguments[0].click();", next_btn)
        time.sleep(1.2)

        driver.execute_script("window.scrollTo(0, 2000);")
        time.sleep(1.0)

        reviews.extend(parse_reviews_from_dom(driver))

    return reviews


# ==========================================================
# 6. 상세 페이지 크롤링
# ==========================================================
def parse_book_detail(driver, url):
    driver.get(url)
    time.sleep(1.5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    def sel(selector):
        node = soup.select_one(selector)
        return node.get_text(strip=True) if node else None

    title = sel("h2.gd_name")
    author = sel("span.gd_auth")
    publisher = sel("span.gd_pub a")
    price = sel("span.nor_price em.yes_m")
    sale_price = sel("span.salec_price em.yes_m")

    img_tag = soup.select_one("img.gImg")
    img_url = img_tag["src"] if img_tag else None

    print(f"📚 리뷰 수집 시작: {title}")

    reviews = fetch_all_reviews(driver)

    return {
        "title": title,
        "author": author,
        "publisher": publisher,
        "price": price,
        "sale_price": sale_price,
        "image": img_url,
        "url": url,
        "reviews": reviews,
    }


# ==========================================================
# 7. 리스트 페이지 크롤링
# ==========================================================
BASE_URL = "https://www.yes24.com/Product/Category/Display/001001008020"

def get_list_page(driver, page):
    url = BASE_URL + f"?page={page}"
    driver.get(url)
    time.sleep(1.5)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.select("a.gd_name")
    urls = ["https://www.yes24.com" + a["href"] for a in items]

    # 총 페이지 가져오기
    pages = soup.select("div.num_pg a")
    max_pages = []
    for p in pages:
        try:
            max_pages.append(int(p.get_text(strip=True)))
        except:
            pass
    total_pages = max(max_pages) if max_pages else 1

    return urls, total_pages


# ==========================================================
# 8. 전체 크롤링 실행
# ==========================================================
def crawl_all_books():
    driver = get_driver()

    print("📖 1페이지 로딩 중…")
    urls, total_pages = get_list_page(driver, 1)

    print(f"🔢 총 {total_pages} 페이지 발견")

    book_urls = urls.copy()

    for p in range(2, total_pages + 1):
        print(f"📄 {p} 페이지 크롤링")
        urls, _ = get_list_page(driver, p)
        book_urls.extend(urls)

    print(f"총 상품 수집: {len(book_urls)}권")

    results = []

    for url in book_urls:
        data = parse_book_detail(driver, url)
        results.append(data)

    driver.quit()
    return results


# ==========================================================
# 9. 저장 (CSV + JSON)
# ==========================================================
def save_results(data):
    # 메타 데이터만 CSV 저장
    rows = []
    for b in data:
        rows.append({
            "title": b["title"],
            "author": b["author"],
            "publisher": b["publisher"],
            "price": b["price"],
            "sale_price": b["sale_price"],
            "image": b["image"],
            "url": b["url"],
            "review_count": len(b["reviews"])
        })
    df = pd.DataFrame(rows)
    df.to_csv("yes24_books.csv", index=False, encoding="utf-8-sig")
    print("📁 yes24_books.csv 저장 완료")

    # 전체 JSON 저장
    with open("yes24_books_full.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print("📁 yes24_books_full.json 저장 완료")


# ==========================================================
# 10. 실행
# ==========================================================
books = crawl_all_books()
save_results(books)

print("🎉 전체 YES24 크롤링 완료!")


📖 1페이지 로딩 중…
🔢 총 1 페이지 발견
총 상품 수집: 24권
📚 리뷰 수집 시작: 연의 편지 (리커버 양장본 한정판 패키지)
📚 리뷰 수집 시작: 만화 나 혼자만 레벨업 15
📚 리뷰 수집 시작: 44교시 생존수업 1~2 세트
🚨 전체 리뷰 탭 클릭 실패: Message: no such element: Unable to locate element: {"method":"css selector","selector":"#total > a > span"}
  (Session info: chrome=142.0.7444.176); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
0   chromedriver                        0x000000010d797698 chromedriver + 6153880
1   chromedriver                        0x000000010d78eb6a chromedriver + 6118250
2   chromedriver                        0x000000010d223a5b chromedriver + 436827
3   chromedriver                        0x000000010d276538 chromedriver + 775480
4   chromedriver                        0x000000010d276791 chromedriver + 776081
5   chromedriver                        0x000000010d2c7934 chromedriver + 1108276
6   chromedriver                        0x000000010d2c4c8e

UnexpectedAlertPresentException: Alert Text: [19세 이상] 나이제한 상품입니다. 로그인 후 이용하세요.
Message: unexpected alert open: {Alert text : [19세 이상] 나이제한 상품입니다. 로그인 후 이용하세요.}
  (Session info: chrome=142.0.7444.176)
Stacktrace:
0   chromedriver                        0x000000010d797698 chromedriver + 6153880
1   chromedriver                        0x000000010d78eb6a chromedriver + 6118250
2   chromedriver                        0x000000010d223a5b chromedriver + 436827
3   chromedriver                        0x000000010d2c55a7 chromedriver + 1099175
4   chromedriver                        0x000000010d268b6f chromedriver + 719727
5   chromedriver                        0x000000010d269871 chromedriver + 723057
6   chromedriver                        0x000000010d754011 chromedriver + 5877777
7   chromedriver                        0x000000010d7584c2 chromedriver + 5895362
8   chromedriver                        0x000000010d730155 chromedriver + 5730645
9   chromedriver                        0x000000010d7

In [76]:
import time
import json
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


# ==========================================================
# 0. Selenium Driver 설정
# ==========================================================
def get_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--window-size=1280,2000")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--ignore-certificate-errors")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=options
    )
    return driver


# ==========================================================
# 1. 성인상품 판별
# ==========================================================
def is_adult_product_soup(soup):
    """
    YES24 상품의 가장 정확한 성인 여부 판별:
    ORD_GOODS_OPT hidden input의 JSON 데이터에서 limit_age 확인
    """

    tag = soup.select_one('input[name="ORD_GOODS_OPT"]')
    if not tag:
        return False

    try:
        data = json.loads(tag["value"])
        if data.get("limit_age_yn") == "Y" and data.get("limit_age") == 19:
            return True
    except:
        pass

    return False


def is_adult_in_list(item_unit):
    """
    리스트 페이지 itemUnit 안에서도 ORD_GOODS_OPT 존재 → 동일하게 판별 가능
    """
    tag = item_unit.select_one('input[name="ORD_GOODS_OPT"]')
    if not tag:
        return False

    try:
        data = json.loads(tag["value"])
        return data.get("limit_age_yn") == "Y" and data.get("limit_age") == 19
    except:
        return False


# ==========================================================
# 2. 리뷰탭 열기
# ==========================================================
def open_review_tab(driver):
    for pos in [300, 700, 1100]:
        driver.execute_script(f"window.scrollTo(0, {pos});")
        time.sleep(0.5)

    try:
        driver.execute_script("goGD_bot(1);")
        time.sleep(1.2)
    except:
        pass


# ==========================================================
# 3. 전체 리뷰 탭 클릭
# ==========================================================
def click_total_review_tab(driver):
    try:
        elem = driver.find_element(By.CSS_SELECTOR, "#total > a > span")
        driver.execute_script("arguments[0].scrollIntoView(true);", elem)
        time.sleep(0.3)
        driver.execute_script("arguments[0].click();", elem)
        time.sleep(1.2)
        return True
    except:
        print("🚨 전체 리뷰 탭 클릭 실패 → 리뷰 없는 상품일 수 있음")
        return False


# ==========================================================
# 4. 리뷰 DOM 파싱
# ==========================================================
def parse_reviews_from_dom(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.select("#infoset_reviewContentList li.reviewItem")

    result = []
    for li in items:
        content = li.select_one("p.review_txt")
        content = content.get_text(strip=True) if content else ""

        user = li.select_one(".txt_userid")
        user = user.get_text(strip=True) if user else ""

        date = li.select_one(".txt_date")
        date = date.get_text(strip=True) if date else ""

        rating = None
        rt = li.select_one("span.rating")
        if rt:
            for cls in rt.get("class", []):
                if "grade_" in cls:
                    rating = cls.replace("grade_", "")
                    break

        result.append({
            "user": user,
            "date": date,
            "rating": rating,
            "content": content,
        })

    return result


# ==========================================================
# 5. 리뷰 전체 페이지 크롤링
# ==========================================================
def fetch_all_reviews(driver):
    open_review_tab(driver)
    click_total_review_tab(driver)

    for pos in [900, 1500, 2000]:
        driver.execute_script(f"window.scrollTo(0, {pos});")
        time.sleep(0.5)

    reviews = []
    reviews.extend(parse_reviews_from_dom(driver))

    while True:
        time.sleep(0.8)

        try:
            paginator = driver.find_element(By.CSS_SELECTOR, ".yesUI_pagenS")
        except:
            break

        soup = BeautifulSoup(driver.page_source, "html.parser")

        try:
            current_page = int(soup.select_one(".yesUI_pagenS strong.num").text.strip())
        except:
            break

        next_btn = None
        for a in driver.find_elements(By.CSS_SELECTOR, ".yesUI_pagenS a.num"):
            if a.text.isdigit() and int(a.text) == current_page + 1:
                next_btn = a
                break

        if not next_btn:
            break

        driver.execute_script("arguments[0].click();", next_btn)
        time.sleep(1.2)
        driver.execute_script("window.scrollTo(0, 2000);")
        time.sleep(1.0)

        reviews.extend(parse_reviews_from_dom(driver))

    return reviews


# ==========================================================
# 6. 상세 페이지 파싱
# ==========================================================
def parse_book_detail(driver, url):
    driver.get(url)
    time.sleep(1.5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # 🔥 성인상품 감지 → 바로 스킵
    if is_adult_product_soup(soup):
        print(f"🚫 성인상품 스킵: {url}")
        return None

    def sel(selector):
        node = soup.select_one(selector)
        return node.get_text(strip=True) if node else None

    title = sel("h2.gd_name")
    author = sel("span.gd_auth")
    publisher = sel("span.gd_pub a")
    price = sel("span.nor_price em.yes_m")
    sale_price = sel("span.salec_price em.yes_m")

    img_tag = soup.select_one("img.gImg")
    img_url = img_tag["src"] if img_tag else None

    print(f"📚 리뷰 수집 시작: {title}")

    reviews = fetch_all_reviews(driver)

    return {
        "title": title,
        "author": author,
        "publisher": publisher,
        "price": price,
        "sale_price": sale_price,
        "image": img_url,
        "url": url,
        "reviews": reviews,
    }


# ==========================================================
# 7. 리스트 페이지 크롤링
# ==========================================================
BASE_URL = "https://www.yes24.com/Product/Category/Display/001001008020"

def get_list_page(driver, page):
    url = BASE_URL + f"?page={page}"
    driver.get(url)
    time.sleep(1.5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    item_units = soup.select(".itemUnit")

    urls = []
    for unit in item_units:
        # 🔥 성인상품 스킵
        if is_adult_in_list(unit):
            title = unit.select_one(".gd_name").get_text(strip=True)
            print(f"🚫 리스트에서 성인상품 스킵: {title}")
            continue

        a = unit.select_one("a.gd_name")
        if a:
            urls.append("https://www.yes24.com" + a["href"])

    # 총 페이지수 계산
    pages = soup.select("div.num_pg a")
    max_pages = []

    for p in pages:
        try:
            max_pages.append(int(p.get_text(strip=True)))
        except:
            pass

    total_pages = max(max_pages) if max_pages else 1

    return urls, total_pages


# ==========================================================
# 8. 전체 크롤링 실행
# ==========================================================
def crawl_all_books():
    driver = get_driver()

    print("📖 1페이지 로딩 중…")
    urls, total_pages = get_list_page(driver, 1)

    print(f"🔢 총 {total_pages} 페이지 발견")

    book_urls = urls.copy()

    for p in range(2, total_pages + 1):
        print(f"📄 {p} 페이지 크롤링")
        urls, _ = get_list_page(driver, p)
        book_urls.extend(urls)

    print(f"📚 크롤링 대상 상품 수: {len(book_urls)}권 (성인상품 자동 제외됨)")

    results = []

    for url in book_urls:
        data = parse_book_detail(driver, url)
        if data is None:
            continue
        results.append(data)

    driver.quit()
    return results


# ==========================================================
# 9. 저장 (CSV + JSON)
# ==========================================================
def save_results(data):
    rows = []
    for b in data:
        rows.append({
            "title": b["title"],
            "author": b["author"],
            "publisher": b["publisher"],
            "price": b["price"],
            "sale_price": b["sale_price"],
            "image": b["image"],
            "url": b["url"],
            "review_count": len(b["reviews"])
        })

    df = pd.DataFrame(rows)
    df.to_csv("yes24_books.csv", index=False, encoding="utf-8-sig")
    print("📁 yes24_books.csv 저장 완료")

    with open("yes24_books_full.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print("📁 yes24_books_full.json 저장 완료")


# ==========================================================
# 10. 실행
# ==========================================================
books = crawl_all_books()
save_results(books)

print("🎉 전체 YES24 크롤링 완료!")


📖 1페이지 로딩 중…
🚫 리스트에서 성인상품 스킵: 시든 꽃에 눈물을 1
🚫 리스트에서 성인상품 스킵: 소꿉친구 컴플렉스 세트
🔢 총 1 페이지 발견
📚 크롤링 대상 상품 수: 22권 (성인상품 자동 제외됨)
📚 리뷰 수집 시작: 연의 편지 (리커버 양장본 한정판 패키지)
📚 리뷰 수집 시작: 만화 나 혼자만 레벨업 15
📚 리뷰 수집 시작: 44교시 생존수업 1~2 세트
🚨 전체 리뷰 탭 클릭 실패 → 리뷰 없는 상품일 수 있음
📚 리뷰 수집 시작: 마루는 강쥐 8
📚 리뷰 수집 시작: 여자친구 1~4권 세트
📚 리뷰 수집 시작: 오래 보고 싶었다
📚 리뷰 수집 시작: 별정직 공무원 2 특별판
🚨 전체 리뷰 탭 클릭 실패 → 리뷰 없는 상품일 수 있음
📚 리뷰 수집 시작: 그저 여명일 뿐 1
📚 리뷰 수집 시작: 만화 인소의 법칙 8 한정판
🚨 전체 리뷰 탭 클릭 실패 → 리뷰 없는 상품일 수 있음
📚 리뷰 수집 시작: 더 트릭컬
📚 리뷰 수집 시작: 그저 여명일 뿐 2
📚 리뷰 수집 시작: 그저 여명일 뿐 3
📚 리뷰 수집 시작: 엄마를 만나러 가는 길 1
📚 리뷰 수집 시작: 발화 1
🚨 전체 리뷰 탭 클릭 실패 → 리뷰 없는 상품일 수 있음
📚 리뷰 수집 시작: 엄마를 만나러 가는 길 2
📚 리뷰 수집 시작: 만화 나 혼자만 레벨업 14
📚 리뷰 수집 시작: 망그러진 만화 (부앙단 댓글 에디션)
📚 리뷰 수집 시작: 호랑이 들어와요 10 특별판
🚨 전체 리뷰 탭 클릭 실패 → 리뷰 없는 상품일 수 있음
📚 리뷰 수집 시작: 고랭순대 작품집 3
🚨 전체 리뷰 탭 클릭 실패 → 리뷰 없는 상품일 수 있음
📚 리뷰 수집 시작: 만화 서울 자가에 대기업 다니는 김 부장 이야기 1~5권 세트
🚨 전체 리뷰 탭 클릭 실패 → 리뷰 없는 상품일 수 있음
📚 리뷰 수집 시작: 역대급 영지 설계사 7
📚 리뷰 수집 시작: 행복아, 어서 와
📁 yes24_books.csv 저장 완료
📁 yes24_books_full.json 저장 완료
🎉 전체 YES24 크롤링 완료!

In [81]:
def debug_reviews_in_detail(detail_url):
    print("\n===== 상세 페이지에서 리뷰 탐색 디버깅 =====")
    resp = requests.get(detail_url, headers=HEADERS)
    print("status:", resp.status_code, "len:", len(resp.text))
    soup = BeautifulSoup(resp.text, "html.parser")

    # 리뷰 영역 근처 텍스트 먼저 찾기
    print("\n--- '회원리뷰' 텍스트 주변 ---")
    around = soup.find(string=lambda x: x and "회원리뷰" in x)
    if around:
        print("회원리뷰 텍스트:", around.strip())
        print("부모 태그:", around.parent)
        print("부모 outerHTML 일부:", str(around.parent)[:500])
    else:
        print("회원리뷰 텍스트 못 찾음")

    # 후보 셀렉터들 한 번씩 다 카운트
    candidates = [
        "div.reviewInfo",
        "div.review_info",
        "div.reviewCont",
        "div.review_cont",
        "li.reviewListItem",
        "li[class*='review']",
        "div[class*='review']",
    ]

    for sel in candidates:
        tags = soup.select(sel)
        print(f"\n[상세페이지 리뷰 후보] {sel} -> {len(tags)}개")
        for t in tags[:2]:
            print("  --- review block ---")
            print(t.get_text(strip=True)[:200])

# 같은 test_detail_url 사용
debug_reviews_in_detail(test_detail_url)



===== 상세 페이지에서 리뷰 탐색 디버깅 =====
status: 200 len: 356905

--- '회원리뷰' 텍스트 주변 ---
회원리뷰 텍스트: 회원리뷰(
부모 태그: <a href="javascript:wiseLog('Pcode','003_006');goGD_bot(1);">회원리뷰(<em class="txC_blue">51</em>건)</a>
부모 outerHTML 일부: <a href="javascript:wiseLog('Pcode','003_006');goGD_bot(1);">회원리뷰(<em class="txC_blue">51</em>건)</a>

[상세페이지 리뷰 후보] div.reviewInfo -> 0개

[상세페이지 리뷰 후보] div.review_info -> 0개

[상세페이지 리뷰 후보] div.reviewCont -> 0개

[상세페이지 리뷰 후보] div.review_cont -> 0개

[상세페이지 리뷰 후보] li.reviewListItem -> 0개

[상세페이지 리뷰 후보] li[class*='review'] -> 0개

[상세페이지 리뷰 후보] div[class*='review'] -> 12개
  --- review block ---
회원리뷰(22건)매주 10건의 우수리뷰를 선정하여 YES포인트 3만원을 드립니다.3,000원 이상 구매 후 리뷰 작성 시일반회원 300원, 마니아회원 600원의 YES포인트를 드립니다.eBook은 다운로드 후 작성한 리뷰만 YES포인트 지급됩니다.클래스는 첫번째 회차 주문확정 시점부터 마지막 회차 주문확정 후 30일 이내 작성한 리뷰만 포인트가 지급됩니다.
  --- review block ---
22명의 예스24 회원이 평가한 평균별점리뷰 총점9.8/ 10.0AI가 리뷰를 요약했어요!AI리뷰 안내AI 리뷰가 도움이 되었나요?좋아요0아쉬워요0별점별로 리뷰를 확인해 보세요.평점 9.0 ~ 10점91%평점 7.0 ~ 8.0점9%평점 5.0 ~ 6.0점0%평점 3.0 ~ 4.0점0%평점 0

In [1]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
BASE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
def crawl_goods(category_name, url, max_pages=1):
    items = []
    for page in range(1, max_pages + 1):
        params = {"PageNumber": page}
        res = requests.get(url, headers=BASE_HEADERS, params=params)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        products = soup.select("ul#yesNewList > li")
        if not products:
            break
        for li in products:
            img_el = li.select_one("img.lazy")
            if not img_el:
                continue
            title_el = li.select_one("div.info_row.info_name > a.gd_name")
            thumb_url = img_el.get("data-original") or img_el.get("src")
            price_el = li.select_one("div.info_row.info_price em.yes_b")
            if not title_el:
                continue
            title = title_el.get_text(strip=True)
            detail_url = urljoin("https://www.yes24.com", title_el.get("href", ""))
            # thumbnail = thumb_el.get("src") if thumb_el else None
            price_text = price_el.get_text(strip=True).replace(",", "") if price_el else ""
            try:
                price = int("".join(ch for ch in price_text if ch.isdigit()))
            except ValueError:
                price = None
            items.append({
                "category": category_name,
                "title": title,
                "price": price,
                "thumbnail": thumb_url,
                "detail_url": detail_url,
            })
    return items
goods = []
goods += crawl_goods("학습/독서", "https://www.yes24.com/product/category/display/006001083", max_pages=1)
goods += crawl_goods("디지털",   "https://www.yes24.com/product/category/display/006001089", max_pages=1)
goods += crawl_goods("디자인문구", "https://www.yes24.com/product/category/display/006001004", max_pages=1)
# len(goods), goods[:3]
print(goods)

[{'category': '학습/독서', 'title': '[예스24배송] 인덱스 마그네틱 북마크', 'price': 1050, 'thumbnail': 'https://image.yes24.com/goods/107977451/L', 'detail_url': 'https://www.yes24.com/product/goods/107977451'}, {'category': '학습/독서', 'title': '★연말특가★[예스24배송]썸라이크 집게 북클립', 'price': 2500, 'thumbnail': 'https://image.yes24.com/goods/144413831/L', 'detail_url': 'https://www.yes24.com/product/goods/144413831'}, {'category': '학습/독서', 'title': '노르잇 투명독서대 높이조절 PR01A', 'price': 28900, 'thumbnail': 'https://image.yes24.com/goods/119691471/L', 'detail_url': 'https://www.yes24.com/product/goods/119691471'}, {'category': '학습/독서', 'title': '[예스24배송]비비드 인덱스 마그네틱 북마크', 'price': 1050, 'thumbnail': 'https://image.yes24.com/goods/123154324/L', 'detail_url': 'https://www.yes24.com/product/goods/123154324'}, {'category': '학습/독서', 'title': '[예스24배송] 감자튀김 모양 과자 밀봉집게 (12개 세트)', 'price': 1600, 'thumbnail': 'https://image.yes24.com/goods/122588631/L', 'detail_url': 'https://www.yes24.com/product/goods/122588631'}, {'category': '학

In [ ]:
import json

with 